# 📈 EPS Forecast Fine-Tuning — DeepSeek-R1-Distill-Llama-8B

**Model:** `unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit`  
**Task:** Top-Down Income Statement EPS Forecasting  
**Dataset:** 677 Alpaca-format financial reasoning samples  

---

## 🗂️ Notebook Phases

| Phase | Description |
|---|---|
| **Phase 1** | Install dependencies |
| **Phase 2** | Imports & configuration |
| **Phase 3** | Load base model + attach LoRA adapters |
| **Phase 4** | Load & reformat dataset (fix revision loop) |
| **Phase 5** | Memory optimisation |
| **Phase 6** | Train |
| **Phase 7** | Save model permanently |

---

### ⚙️ Kaggle Setup Checklist
- [ ] **Accelerator:** GPU T4 x2
- [ ] **Internet:** ON
- [ ] Dataset uploaded at: `/kaggle/input/<your-slug>/final_verified_dataset (1).jsonl`
- [ ] Update `DATASET_PATH` in Phase 2 before running

---
## ⚡ Phase 1 — Install Dependencies

Installs `unsloth` (fast fine-tuning library) and then force-reinstalls the latest nightly build from GitHub to ensure all patches are applied. The `%%capture` suppresses the lengthy pip output — remove it if you want to see install logs.

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
print('✅ Unsloth installed')

---
## 📦 Phase 2 — Imports & Configuration

All imports are consolidated here so nothing is missing when cells are re-run out of order.

### Key configuration decisions

| Parameter | Value | Reason |
|---|---|---|
| `MAX_SEQ_LEN` | 2048 | Financial inputs + full analysis = 800–1200 tokens |
| `LORA_RANK` | 32 | Higher rank = more capacity to learn the stop pattern |
| `EPOCHS` | 5 | Needs more passes to unlearn the revision loop from dataset |
| `LR` | 1e-4 | Lower than default — finer updates prevent catastrophic forgetting |
| `BATCH_SIZE` | 1 | T4 has 14.5 GB — 2048 seq len + rank 32 needs conservative batch |
| `GRAD_ACCUM` | 16 | Effective batch = 1×16 = 16, same as before |

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import gc

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"
DATASET_PATH = "/kaggle/input/datasets/mhariyan/eps-forecast-dataset/final_verified_dataset (1).jsonl"  # ← UPDATE THIS
OUTPUT_DIR   = "/kaggle/working/eps-forecast-lora"
# ADAPTER_PATH = kagglehub.model_download("mhariyan/eps_model/keras/default3")

MAX_SEQ_LEN  = 2048   # full financial analysis needs this
LORA_RANK    = 32     # ↑ from 16 — more capacity for stop-signal learning
EPOCHS       = 7      # ↑ from 3 — needs more passes to fix revision-loop behaviour
LR           = 1e-4   # ↓ from 2e-4 — finer updates with more epochs
BATCH_SIZE   = 1      # conservative for 2048 seq len on T4
GRAD_ACCUM   = 16     # effective batch = 16

print(f"✅ Configuration loaded")
print(f"   Model       : {MODEL_NAME}")
print(f"   Max seq len : {MAX_SEQ_LEN}")
print(f"   LoRA rank   : {LORA_RANK}")
print(f"   Epochs      : {EPOCHS}")
print(f"   LR          : {LR}")
print(f"   Eff. batch  : {BATCH_SIZE * GRAD_ACCUM}")

---
## 🤖 Phase 3 — Load Base Model + Attach LoRA Adapters

### What happens here
1. **Load base model** — `DeepSeek-R1-Distill-Llama-8B` in 4-bit quantisation. This uses ~5 GB of the T4's 14.5 GB VRAM.
2. **Attach LoRA adapters** — Only 0.5% of weights are trainable. The rest are frozen, which is what makes fine-tuning feasible on a T4.

### LoRA target modules explained
We target all projection layers (`q/k/v/o`) and the MLP layers (`gate/up/down`). This gives the model maximum ability to learn the new output format without overfitting.

---
## 🗃️ Phase 4 — Load & Reformat Dataset

### The core problem this phase fixes

Every sample in the original dataset ends with a **revision loop**:
```
Step 6: Final Calculation
EPS = 32.42
However, this doesn't align with our target. Let's revisit...  ← model learns to keep going
adjust margins...
Thus, the forecasted...  ← still going

Forecast: 18.90   ← buried 800 tokens in, after multiple revisions
```

The model learned from this that **after computing EPS, you keep revising**. So at inference time it generates endlessly and never stops at `Forecast:`.

### The fix
`reformat_output()` strips everything after Step 6 and replaces it with a single clean line:
```
Step 6: Final Calculation
Forecast: 18.90<EOS>   ← model learns to STOP here
```

The `EOS_TOKEN` placed immediately after `Forecast: XX.XX` is critical — it teaches the model that this is a **terminal signal**, not a line to continue from.

In [ ]:
# ── Load base model ───────────────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,        # auto-detect: float16 on T4
    load_in_4bit    = True,
)

# ── Attach LoRA adapters ──────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_RANK,    # keep equal to rank
    target_modules             = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout               = 0.05,         # small dropout prevents overfitting
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",    # saves ~30% VRAM
    random_state               = 42,
)

EOS_TOKEN = tokenizer.eos_token

print(f"\n✅ Model loaded")
print(f"   EOS token : {repr(EOS_TOKEN)}")
model.print_trainable_parameters()

In [ ]:
# ── Strict training prompt ─────────────────────────────────────────────────────
# FORECAST: appears as its own dedicated line — not mixed into Step 6 text
ALPACA_PROMPT_TRAIN = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
FORECAST: {forecast_value}"""


def reformat_output(output_text: str):
    """
    Strips everything from Step 6 onwards.
    Keeps only Steps 1-5 reasoning.
    Injects FORECAST: as a standalone final line via the prompt template.

    Training sample will look like:
        Step 1: Revenue Analysis...
        Step 2: Operating Margin...
        Step 3: Interest & Tax...
        Step 4: Net Income...
        Step 5: Share Count...
        FORECAST: 47.68<EOS>
    """
    # Find the LAST forecast value — always the ground truth
    matches = list(re.finditer(
        r'\bforecast\b\s*[:\=]\s*(-?\d+(?:\.\d+)?)',
        output_text, re.IGNORECASE
    ))
    if not matches:
        return None, None

    forecast_value = matches[-1].group(1)

    # Cut everything from Step 6 / Final Calculation onwards
    step6 = re.search(r'(step\s*6|final\s*calculation)', output_text, re.IGNORECASE)
    if step6:
        body = output_text[:step6.start()].rstrip()
    else:
        # No Step 6 — cut just before the last forecast line
        body = output_text[:matches[-1].start()].rstrip()

    return body, forecast_value


def formatting_prompts_func(examples):
    texts   = []
    skipped = 0

    for instruction, inp, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        body, forecast_value = reformat_output(output)

        if body is None:
            skipped += 1
            continue

        text = ALPACA_PROMPT_TRAIN.format(
            instruction    = instruction,
            input          = inp,
            output         = body,
            forecast_value = forecast_value,
        ) + EOS_TOKEN   # EOS immediately after FORECAST: XX.XX

        texts.append(text)

    return {"text": texts}


# ── Load raw dataset ──────────────────────────────────────────────────────────
raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Raw samples loaded  : {len(raw)}")

# ── Apply reformatting ────────────────────────────────────────────────────────
dataset = raw.map(formatting_prompts_func, batched=True, remove_columns=raw.column_names)
print(f"Formatted samples   : {len(dataset)}")
print(f"Skipped (no GT)     : {len(raw) - len(dataset)}")

# ── Verify the format looks correct ──────────────────────────────────────────
sample = dataset[0]["text"]
print("\n--- Last 200 chars of sample 0 ---")
print(sample[-200:])
print("\n✅ Dataset ready — confirm FORECAST: appears at the very end above")

---
## 🧠 Phase 5 — Memory Optimisation

Before training, free any GPU memory from the model load and dataset processing steps. This gives the trainer maximum headroom and reduces the chance of OOM errors.

`expandable_segments=True` tells PyTorch to reuse memory blocks more aggressively, which prevents fragmentation during the backward pass.

In [ ]:
# ── Free leftover GPU memory ──────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# ── Smarter memory allocation ─────────────────────────────────────────────────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

free_gb  = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9

print(f"✅ Memory optimised")
print(f"   Free  : {free_gb:.2f} GB")
print(f"   Total : {total_gb:.2f} GB")
print(f"   Used  : {total_gb - free_gb:.2f} GB")

---
## 🚀 Phase 6 — Train

### SFTConfig decisions

| Parameter | Value | Reason |
|---|---|---|
| `packing=False` | False | With 2048 seq len, packing risks OOM |
| `average_tokens_across_devices=False` | False | Prevents `AttributeError: int has no .mean()` on multi-GPU |
| `gradient_checkpointing=True` | True | Recomputes activations on backward — saves ~2 GB VRAM |
| `dataloader_pin_memory=False` | False | Saves ~500 MB pinned RAM |
| `warmup_steps=20` | 20 | Scaled up from 10 since we have 5 epochs |
| `lr_scheduler_type=cosine` | cosine | Smooth LR decay across 5 epochs |

### Expected training time
~3–4 hours on T4 for 5 epochs over 669 samples.

In [ ]:
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        # ── Data ──────────────────────────────────────────────────────────────
        dataset_text_field            = "text",
        max_seq_length                = MAX_SEQ_LEN,
        dataset_num_proc              = 2,
        packing                       = False,      # off — 2048 seq len + OOM risk

        # ── Training ──────────────────────────────────────────────────────────
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRAD_ACCUM,
        num_train_epochs              = EPOCHS,
        learning_rate                 = LR,
        warmup_steps                  = 20,
        lr_scheduler_type             = "cosine",
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        seed                          = 42,

        # ── Precision ─────────────────────────────────────────────────────────
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),

        # ── Memory ────────────────────────────────────────────────────────────
        gradient_checkpointing        = True,
        dataloader_pin_memory         = False,
        average_tokens_across_devices = False,      # prevents int .mean() error

        # ── Logging & Saving ──────────────────────────────────────────────────
        logging_steps                 = 10,
        save_strategy                 = "epoch",
        output_dir                    = OUTPUT_DIR,
    ),
)

print("✅ Trainer configured — starting training...")
print(f"   Steps per epoch : {len(dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps     : {len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}")
print()

In [ ]:
trainer_stats = trainer.train()

peak_vram = torch.cuda.max_memory_reserved() / 1e9
print(f"\n✅ Training complete")
print(f"   Peak VRAM used  : {peak_vram:.2f} GB")
print(f"   Training loss   : {trainer_stats.training_loss:.4f}")

---
## 💾 Phase 7 — Save Model

Kaggle's `/kaggle/working/` is **wiped after every session**. You must save to a permanent location before the session ends.

### What gets saved

| Method | Size | What it contains |
|---|---|---|
| **LoRA adapters only** | ~100–200 MB | Just the trained weight deltas |
| **Merged 16-bit model** | ~15 GB | Full standalone model |
| **Kaggle Models upload** | ~100–200 MB | Permanent, reusable in future notebooks |

### ⚠️ Important
Always save LoRA adapters — not the full merged model. At inference time, load the base model + adapters together. This keeps your upload small and fast.

In [ ]:
# ── Step 1: Save LoRA adapters locally ───────────────────────────────────────
LORA_SAVE_PATH = "/kaggle/working/eps-lora-adapters"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

# Show what was saved
import os
files = os.listdir(LORA_SAVE_PATH)
total_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_PATH, f))
    for f in files
) / 1e6

print(f"✅ LoRA adapters saved to: {LORA_SAVE_PATH}")
print(f"   Files   : {files}")
print(f"   Total   : {total_mb:.1f} MB")

In [ ]:
# ── Step 2: Upload to Kaggle Models (permanent storage) ──────────────────────
import kagglehub

MODEL_SLUG     = "eps_model"    # ← your model name on Kaggle
VARIATION_SLUG = "default3"     # ← increment each retrain
HF_USERNAME    = "mhariyan"     # ← your Kaggle username

kagglehub.model_upload(
    handle          = f"{HF_USERNAME}/{MODEL_SLUG}/keras/{VARIATION_SLUG}",   # ← fixed
    local_model_dir = LORA_SAVE_PATH,
    version_notes   = "v3 — FORECAST: terminal, revision loop stripped, outliers filtered, H100 optimised, 7 epochs"
)

print(f"✅ Model uploaded to Kaggle")
print(f"   Handle : {HF_USERNAME}/{MODEL_SLUG}/keras/{VARIATION_SLUG}")
print(f"   Load in future notebooks with:")
print(f"   kagglehub.model_download('{HF_USERNAME}/{MODEL_SLUG}/keras/{VARIATION_SLUG}')")

---
## ✅ Training Complete

### What changed vs previous runs

| What | Old | New | Impact |
|---|---|---|---|
| `LORA_RANK` | 4 → 16 | **32** | More capacity to learn stop pattern |
| `EPOCHS` | 1 → 3 | **5** | More passes over the cleaned data |
| `LR` | 2e-4 | **1e-4** | Finer updates, less catastrophic forgetting |
| `lora_dropout` | 0 | **0.05** | Prevents overfitting on 669 samples |
| `MAX_SEQ_LEN` | 512 | **2048** | Full financial analysis fits in context |
| Dataset outputs | Revision loop | **Clean `Forecast: XX.XX<EOS>`** | Model stops at forecast line |

### Next step — Inference
Load `default2` in your inference notebook:
```python
MODEL_PATH = kagglehub.model_download('mhariyan/eps_model/keras/default2')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit   = True,
)
```